# Student Dropout Prediction — Preprocessing

This notebook prepares the dataset for model training by filtering the target classes, removing academic-related features, and encoding the target variable.

## Import Library

In [44]:
from pathlib import Path

import pandas as pd
import numpy as np

## Load Data

In [45]:
current_path = Path.cwd()

if (current_path / "data" / "raw" / "dataset.csv").exists():
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path.parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "dataset.csv"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "student_dropout_non_academic_binary.csv"

print("Project root:", PROJECT_ROOT)
print("Raw data path:", RAW_DATA_PATH)
print("Raw dataset exists:", RAW_DATA_PATH.exists())
print("Processed data path:", PROCESSED_DATA_PATH)

Project root: c:\Not Default\Projects\student-dropout-prediction-ml
Raw data path: c:\Not Default\Projects\student-dropout-prediction-ml\data\raw\dataset.csv
Raw dataset exists: True
Processed data path: c:\Not Default\Projects\student-dropout-prediction-ml\data\processed\student_dropout_non_academic_binary.csv


In [46]:
df = pd.read_csv(RAW_DATA_PATH)

print("Raw dataset loaded successfully.")
print("Raw dataset shape:", df.shape)

df.head()

Raw dataset loaded successfully.
Raw dataset shape: (4424, 35)


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Nacionality,Mother's qualification,Father's qualification,Mother's occupation,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,8,5,2,1,1,1,13,10,6,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,6,1,11,1,1,1,1,3,4,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,5,1,1,1,22,27,10,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,8,2,15,1,1,1,23,27,6,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,12,1,3,0,1,1,22,28,10,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


## Preprocessing

### Change Data

In [47]:
# Remove Enrolled Data
df_processed = df[df["Target"].isin(["Graduate", "Dropout"])].copy()

print("Raw dataset shape:", df.shape)
print("Processed dataset shape after removing Enrolled:", df_processed.shape)
print("Removed Enrolled rows:", df.shape[0] - df_processed.shape[0])

df_processed["Target"].value_counts() # Only Graduate and Droput in Target

Raw dataset shape: (4424, 35)
Processed dataset shape after removing Enrolled: (3630, 35)
Removed Enrolled rows: 794


Target
Graduate    2209
Dropout     1421
Name: count, dtype: int64

In [48]:
# Remove Academic Features
academic_semester_features = [
    col for col in df_processed.columns
    if "Curricular units" in col
]

academic_grade_features = [
    "Admission grade",
    "Previous qualification"
]

academic_grade_features = [
    col for col in academic_grade_features
    if col in df_processed.columns
]

academic_features = academic_semester_features + academic_grade_features

print("Academic semester features:", len(academic_semester_features))
print("Academic grade features:", len(academic_grade_features))
print("Total academic features to remove:", len(academic_features))

df_processed = df_processed.drop(columns=academic_features).copy()

academic_features

Academic semester features: 12
Academic grade features: 1
Total academic features to remove: 13


['Curricular units 1st sem (credited)',
 'Curricular units 1st sem (enrolled)',
 'Curricular units 1st sem (evaluations)',
 'Curricular units 1st sem (approved)',
 'Curricular units 1st sem (grade)',
 'Curricular units 1st sem (without evaluations)',
 'Curricular units 2nd sem (credited)',
 'Curricular units 2nd sem (enrolled)',
 'Curricular units 2nd sem (evaluations)',
 'Curricular units 2nd sem (approved)',
 'Curricular units 2nd sem (grade)',
 'Curricular units 2nd sem (without evaluations)',
 'Previous qualification']

### Encode Data

In [49]:
# Encode target variable directly
target_mapping = {
    "Graduate": 0,
    "Dropout": 1
}

df_processed["Target"] = df_processed["Target"].map(target_mapping)

# Convert target to integer
df_processed["Target"] = df_processed["Target"].astype(int)

# Validate encoded target
print("Missing values in Target:", df_processed["Target"].isnull().sum())

df_processed["Target"].value_counts().sort_index()

Missing values in Target: 0


Target
0    2209
1    1421
Name: count, dtype: int64

## Save Data

In [50]:
# Save processed dataset
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "processed.csv"

# Create processed folder if it does not exist
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

# Save and replace existing file if it already exists
df_processed.to_csv(PROCESSED_DATA_PATH, index=False)

# Validate saved file
print("Processed dataset saved to:", PROCESSED_DATA_PATH)
print("File exists:", PROCESSED_DATA_PATH.exists())
print("Saved dataset shape:", df_processed.shape)

Processed dataset saved to: c:\Not Default\Projects\student-dropout-prediction-ml\data\processed\processed.csv
File exists: True
Saved dataset shape: (3630, 22)
